# 🎬 X-Studio: Colab-Accelerated AI Video Generator

**Developer:** [@SILENTXOP](https://github.com/silentxop)  
**YouTuber:** [@silentx_nomore](https://youtube.com/@silentx_nomore)  
**YouTube Channel:** [https://youtube.com/@silentx_nomore](https://youtube.com/@silentx_nomore)  

---

### 📌 Target Architecture
Windows Low-End PC ➔ Chrome Browser ➔ X-Studio Web UI ➔ Google Colab GPU ➔ Local Model Synthesis ➔ Direct MP4 Download to Windows PC.

> **Important:** 
> 1. Make sure you are using a GPU runtime: **Runtime** -> **Change runtime type** -> **T4 GPU** (or L4 / A100).
> 2. No Google Drive required — video files are saved in Colab temporary storage and downloaded directly via browser.
> 3. Low-end PC friendly — zero local GPU required on your Windows computer.

## 🔍 Step 1: Verify NVIDIA GPU & CUDA Environment
Run this cell to detect your GPU model, available VRAM, and verify CUDA.

In [ ]:
import torch
import sys

print('=' * 60)
print('🔍 X-Studio Hardware Diagnostic')
print('=' * 60)

if not torch.cuda.is_available():
    raise SystemExit(
        '❌ CRITICAL: No GPU detected!\n'
        'Please switch to a GPU runtime in Colab:\n'
        '1. Click Runtime in the top menu.\n'
        '2. Select Change runtime type.\n'
        '3. Under Hardware accelerator, select T4 GPU (free) or L4/A100.\n'
        '4. Click Save and re-run this cell.'
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f'✅ GPU Detected:     {gpu_name}')
print(f'✅ Total VRAM:       {total_vram_gb:.2f} GB')
print(f'✅ PyTorch Version:  {torch.__version__}')
print(f'✅ CUDA Version:     {torch.version.cuda}')

if total_vram_gb >= 14.0:
    print('\n🚀 Recommended Models: Wan 2.1 (1.3B) [Fast, High Quality], CogVideoX (2B), LTX-2, Wan 2.2 TI2V (5B)')
elif total_vram_gb >= 7.0:
    print('\n🚀 Recommended Models: Wan 2.1 (1.3B), CogVideoX (2B)')
else:
    print('\n🚀 Recommended Models: CogVideoX (2B) [Ultra-low VRAM]')
print('=' * 60)

## 📦 Step 2: Install X-Studio Dependencies
Clone or sync the repository and install required packages (`diffusers`, `transformers`, `accelerate`, `gradio`, etc.).

In [ ]:
import os, sys

# Clone repo if not already present
if not os.path.exists('XC-Studio') and not os.path.exists('x_studio_engine.py'):
    print('📥 Cloning X-Studio repository...')
    !git clone https://github.com/xcode8908/XC-Studio.git
    %cd XC-Studio
elif os.path.exists('XC-Studio'):
    %cd XC-Studio
    !git pull

print('\n📦 Installing required Python dependencies...')
!pip install -q diffusers transformers accelerate sentencepiece gradio imageio imageio-ffmpeg pydantic pyyaml requests

print('\n✅ Environment ready for X-Studio generation!')

## 🚀 Step 3: Launch X-Studio Web UI
Run this cell to start the X-Studio server.  
Look for the **`Running on public URL: https://xxxx.gradio.live`** in the output and click it to control X-Studio from **Google Chrome** on your Windows PC!

In [ ]:
from x_studio_server import start_server

# Starts server and provides a public HTTPS link for your Chrome browser
start_server(port=7860, share=True)

## 🧹 Optional: Free Colab Disk Space (If Disk Warning Appears)
If Google Colab displays a 'Disk is almost full' warning, run this cell to instantly clear cached packages and restore 20-30 GB free space.

In [ ]:
!rm -rf /root/.cache/huggingface /root/.cache/pip /tmp/*
!pip cache purge
!apt-get clean
print('✅ Disk cleanup completed! Free disk space restored.')

## 🧪 Optional: Quick Test Generation via Python Script
If you prefer to test video generation directly without launching the UI, run this cell.

In [ ]:
from x_studio_engine import generate_video

test_prompt = 'A cinematic drone shot flying over a majestic medieval castle on a misty green mountain at sunrise, golden sunlight rays, volumetric clouds, photorealistic, 4k movie trailer style'
print(f"🎬 Generating CogVideoX (2B) video: '{test_prompt}'...")

result = generate_video(
    prompt=test_prompt,
    model_key='cogvideo-2b',  # Ultra-lightweight & fits comfortably in Colab disk
    resolution='720x480',
    duration_seconds=3.0,
    steps=20,
    seed=42,
)

if result['success']:
    print(f'\n✅ Test video generated successfully!')
    print(f"Output path: {result['output_path']}")
    print(f"Duration:    {result['duration']}s")
    print(f"File Size:   {result['file_size_mb']} MB")
else:
    print(f"\n❌ Generation failed: {result['error']}")